# X02 · Compute budgets, roofline, latency and energy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/10_compute_deployment/X02_compute_budgets_roofline_latency_and_energy.ipynb)

**Track:** Compute and deployment · **Skill axis (sheet):** Compute and deployment · **Time:** about 55 min

**Skill Hub concepts:** Multiply-accumulate; Arithmetic intensity and roofline; Latency versus throughput; Energy per inference

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("X02")

## Map: will this model fit on that chip, in time, on battery?

**Multiply-accumulates (MACs)** per layer (1 MAC = 2 floating-point operations):
$$\text{Conv2d: } C_{out} H_{out} W_{out} \cdot \frac{C_{in}}{g} \cdot k_h k_w \qquad \text{Linear: } F_{in} F_{out} \qquad (g = \text{groups; depthwise: } g = C_{in})$$
**Arithmetic intensity** $I$ = FLOPs per byte moved to and from memory. **Roofline:** attainable throughput is
$$P = \min\left(P_{peak},\ I \cdot BW\right), \qquad \text{ridge point } I^* = P_{peak} / BW$$
Below the ridge you are **memory-bound** (faster memory or more reuse helps); above it, **compute-bound**.
Matrix multiply $M \times K$ by $K \times N$: $I = \dfrac{2MNK}{b\,(MK + KN + MN)}$ with $b$ bytes per element.

| Metric | Formula | Who cares |
|---|---|---|
| Latency | time for one request | closed-loop control: every millisecond is phase lag |
| Throughput | batch / latency | offline processing, many cameras |
| Energy per inference | power x latency / batch | battery robots, thermal limits |

Batching raises throughput (better weight reuse, higher intensity) but raises latency: a control loop wants batch 1.

In [ ]:
import time
import numpy as np
import torch

## Task 1: count MACs analytically and with forward hooks
`model_macs` runs one forward pass on `torch.zeros(input_shape)` (batch 1) and sums Conv2d and Linear MACs using
forward hooks (`module.register_forward_hook`); read the output shape inside the hook.

In [ ]:
def conv2d_macs(cin, cout, k, hout, wout, groups=1):
    raise NotImplementedError  # TODO

def linear_macs(fin, fout):
    raise NotImplementedError  # TODO

def model_macs(model, input_shape):
    raise NotImplementedError  # TODO

lab.check("macs", conv2d_macs, linear_macs, model_macs)

## Task 2: arithmetic intensity and the roofline

In [ ]:
def matmul_intensity(M, N, K, bytes_per_element):
    raise NotImplementedError  # TODO

def attainable(peak_gflops, bandwidth_gbs, intensity):
    raise NotImplementedError  # TODO

lab.check("roofline", matmul_intensity, attainable)

## Task 3: throughput and energy

In [ ]:
def throughput(batch, latency_s):
    raise NotImplementedError  # TODO

def energy_per_inference(power_w, latency_s, batch=1):
    raise NotImplementedError  # TODO

lab.check("energy", throughput, energy_per_inference)

Measure it (not graded): latency and throughput of a small CNN on this machine versus batch size.

In [ ]:
cnn = torch.nn.Sequential(torch.nn.Conv2d(3, 32, 3, padding=1), torch.nn.ReLU(), torch.nn.Conv2d(32, 64, 3, stride=2, padding=1),
                          torch.nn.ReLU(), torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(), torch.nn.Linear(64, 10)).eval()
try:
    print(f"MACs per image: {model_macs(cnn, (1, 3, 64, 64)):,}")
    for b in [1, 4, 16, 64]:
        x = torch.randn(b, 3, 64, 64)
        with torch.no_grad():
            cnn(x); t = time.perf_counter()
            for _ in range(5): cnn(x)
        lat = (time.perf_counter() - t) / 5
        print(f"batch {b:3d}: latency {lat * 1e3:7.2f} ms, throughput {throughput(b, lat):8.0f} images/s")
except NotImplementedError:
    print("finish Tasks 1 and 3 first")

## Checkpoint: Math
**M1 (Apply).** FLOPs (2 per MAC) of a 3 x 3 convolution, 64 to 64 channels, output 56 x 56?

**M2 (Apply).** Memory bandwidth 100 GB/s, intensity 10 FLOP/byte, peak 2000 GFLOP/s. Attainable GFLOP/s?

**M3 (Apply).** A 5 W accelerator takes 20 ms per image at batch 1. Joules per inference?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A 4096 x 4096 fully connected layer at batch 1 in float16 on a GPU is:
- a) memory-bound: about 1 FLOP per byte, far below the ridge point
- b) compute-bound: matrix multiplies always are
- c) neither: it is latency-bound by the kernel launch only

**K2 (Evaluate).** A perception model runs at 120 images/s with batch 8 or 60 images/s with batch 1. Which do you deploy
inside a 30 Hz visual servoing loop, and why?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `x02.h`).
3. In `roboprep/checks/x02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()